# HMM-based Noun Phrase (NP) Chunking
States: `B-NP`, `I-NP`, `O`. Observations: words.

1. **Forward algorithm** – probability `P(sentence)` of each test sentence.
2. **Viterbi decoding** – most likely tag sequence, evaluated with `seqeval`.

Zero counts are handled with **Add-K smoothing** (transitions, start probabilities and emissions; unseen words map to an `<UNK>` emission). All computations are done in log space to avoid underflow.

In [6]:
!pip install seqeval
import numpy as np
from collections import Counter, defaultdict
from scipy.special import logsumexp
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

TRAIN_PATH = '/Users/mjgandhi2305/Semester_7/ILLM/Lab7/Data/test_NP.txt'   # change paths if needed
TEST_PATH  = '/Users/mjgandhi2305/Semester_7/ILLM/Lab7/Data/test_NP.txt'

## 1. Load data
Format: `word<TAB>tag`, sentences separated by blank lines.

In [7]:
def read_data(path):
    sents, words, tags = [], [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if words:
                    sents.append((words, tags)); words, tags = [], []
                continue
            parts = line.split()
            words.append(parts[0]); tags.append(parts[-1])
    if words: sents.append((words, tags))
    return sents

train = read_data(TRAIN_PATH)
test  = read_data(TEST_PATH)
print('train sentences:', len(train), '| test sentences:', len(test))
print(train[0])

train sentences: 77 | test sentences: 77
(['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.'], ['B-NP', 'O', 'B-NP', 'I-NP', 'O', 'O', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'O', 'O', 'O', 'B-NP', 'B-NP', 'O', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O'])


## 2. Estimate HMM parameters (with Add-K smoothing)
- Start: `π(t) = (C(start→t)+K) / (N_sent + K·|T|)`
- Transition: `A(t'|t) = (C(t→t')+K) / (C(t)+K·|T|)`
- Emission: `B(w|t) = (C(t,w)+K) / (C(t)+K·(|V|+1))`; unseen words use the `<UNK>` slot, so every word has non-zero probability.

Words are lower-cased for emission; this reduces sparsity.

In [8]:
K = 0.01
tags = sorted({t for _, ts in train for t in ts})
tag2i = {t:i for i,t in enumerate(tags)}
T = len(tags)
norm = lambda w: w.lower()
vocab = {norm(w) for ws,_ in train for w in ws}
V = len(vocab)

start_c = np.zeros(T); trans_c = np.zeros((T,T)); tag_c = np.zeros(T)
emit_c = defaultdict(lambda: np.zeros(T))
for ws, ts in train:
    start_c[tag2i[ts[0]]] += 1
    for j,(w,t) in enumerate(zip(ws,ts)):
        ti = tag2i[t]; tag_c[ti]+=1; emit_c[norm(w)][ti]+=1
        if j>0: trans_c[tag2i[ts[j-1]], ti]+=1

log_pi = np.log((start_c+K)/(start_c.sum()+K*T))
log_A  = np.log((trans_c+K)/(trans_c.sum(1,keepdims=True)+K*T))
emit_den = tag_c + K*(V+1)
log_B  = {w: np.log((c+K)/emit_den) for w,c in emit_c.items()}
log_B_unk = np.log(K/emit_den)

def emis(w):  # vector of log B(w|t) over tags
    return log_B.get(norm(w), log_B_unk)

print('tags:', tags, '| vocab size:', V)
print('Start probs:', dict(zip(tags, np.exp(log_pi).round(4))))
print('Transition matrix (rows=from):'); print(np.exp(log_A).round(4))

tags: ['B-NP', 'I-NP', 'O'] | vocab size: 702
Start probs: {'B-NP': 0.6492, 'I-NP': 0.0001, 'O': 0.3506}
Transition matrix (rows=from):
[[0.0127 0.7094 0.2779]
 [0.0433 0.4187 0.5381]
 [0.5144 0.     0.4856]]


## 3. Forward algorithm
`α₁(j)=π(j)B(w₁|j)`, `α_t(j)=Σᵢ α_{t-1}(i)A(j|i)·B(w_t|j)`, `P(O)=Σⱼ α_T(j)` (computed in log space with `logsumexp`).

In [9]:
def forward_log_prob(words):
    alpha = log_pi + emis(words[0])
    for w in words[1:]:
        alpha = logsumexp(alpha[:,None] + log_A, axis=0) + emis(w)
    return logsumexp(alpha)

fwd = [forward_log_prob(ws) for ws,_ in test]
print(f'{"#":>3} {"log P(sentence)":>16} {"P(sentence)":>14}  sentence')
for i,(lp,(ws,_)) in enumerate(zip(fwd,test[:10])):
    print(f'{i+1:>3} {lp:16.4f} {np.exp(lp):14.4e}  {" ".join(ws)[:70]}')

  #  log P(sentence)    P(sentence)  sentence
  1        -206.4666     2.1514e-90  Confidence in the pound is widely expected to take another sharp dive 
  2        -146.8675     1.6453e-64  Chancellor of the Exchequer Nigel Lawson 's restated commitment to a f
  3        -168.5392     6.3732e-74  But analysts reckon underlying support for sterling has been eroded by
  4        -185.9650     1.7235e-81  This has increased the risk of the government being forced to increase
  5        -165.8937     8.9802e-73  `` The risks for sterling of a bad trade figure are very heavily on th
  6        -193.7339     7.2850e-85  `` If there is another bad trade number , there could be an awful lot 
  7        -208.7106     2.2810e-91  Forecasts for the trade figures range widely , but few economists expe
  8        -131.4513     8.1549e-58  The August deficit and the # 2.2 billion gap registered in July are to
  9        -163.2068     1.3188e-71  Sanjay Joshi , European economist at Baring Brothers 

## 4. Viterbi decoding

In [10]:
def viterbi(words):
    n = len(words)
    delta = np.zeros((n,T)); psi = np.zeros((n,T), dtype=int)
    delta[0] = log_pi + emis(words[0])
    for t in range(1,n):
        scores = delta[t-1][:,None] + log_A
        psi[t] = scores.argmax(0)
        delta[t] = scores.max(0) + emis(words[t])
    path = [int(delta[-1].argmax())]
    for t in range(n-1,0,-1):
        path.append(int(psi[t,path[-1]]))
    return [tags[i] for i in reversed(path)]

y_true = [ts for _,ts in test]
y_pred = [viterbi(ws) for ws,_ in test]
for ws,tt,pp in list(zip([w for w,_ in test],y_true,y_pred))[:2]:
    print(*[f'{w}/{p}' + ('' if p==t else f'(gold {t})') for w,t,p in zip(ws,tt,pp)])
    print()

Confidence/B-NP in/O the/B-NP pound/I-NP is/O widely/O expected/O to/O take/O another/B-NP sharp/I-NP dive/I-NP if/O trade/B-NP figures/I-NP for/O September/B-NP ,/O due/O for/O release/B-NP tomorrow/B-NP ,/O fail/O to/O show/O a/B-NP substantial/I-NP improvement/I-NP from/O July/B-NP and/I-NP August/I-NP 's/B-NP near-record/I-NP deficits/I-NP ./O

Chancellor/O of/O the/B-NP Exchequer/I-NP Nigel/O Lawson/O 's/B-NP restated/I-NP commitment/I-NP to/O a/B-NP firm/O(gold I-NP) monetary/B-NP(gold I-NP) policy/I-NP has/O helped/O to/O prevent/O a/B-NP freefall/I-NP in/O sterling/B-NP over/O the/B-NP past/I-NP week/I-NP ./O



## 5. Evaluation (seqeval)

In [11]:
print(classification_report(y_true, y_pred, digits=4))
print('Chunk-level  P: %.4f  R: %.4f  F1: %.4f' % (precision_score(y_true,y_pred), recall_score(y_true,y_pred), f1_score(y_true,y_pred)))
print('Token accuracy: %.4f' % accuracy_score(y_true, y_pred))

              precision    recall  f1-score   support

          NP     0.9236    0.9411    0.9322       475

   micro avg     0.9236    0.9411    0.9322       475
   macro avg     0.9236    0.9411    0.9322       475
weighted avg     0.9236    0.9411    0.9322       475

Chunk-level  P: 0.9236  R: 0.9411  F1: 0.9322
Token accuracy: 0.9768


## 6. Optional: effect of K
Small sweep over the smoothing constant.

In [12]:
def build(Kv):
    global log_pi, log_A, log_B, log_B_unk
    log_pi = np.log((start_c+Kv)/(start_c.sum()+Kv*T))
    log_A  = np.log((trans_c+Kv)/(trans_c.sum(1,keepdims=True)+Kv*T))
    den = tag_c + Kv*(V+1)
    log_B  = {w: np.log((c+Kv)/den) for w,c in emit_c.items()}
    log_B_unk = np.log(Kv/den)

for Kv in [1, 0.1, 0.01, 0.001, 1e-4]:
    build(Kv)
    pred = [viterbi(ws) for ws,_ in test]
    print(f'K={Kv:<7} F1={f1_score(y_true,pred):.4f}  token-acc={accuracy_score(y_true,pred):.4f}')
build(K)  # restore

K=1       F1=0.7983  token-acc=0.9299
K=0.1     F1=0.9142  token-acc=0.9678
K=0.01    F1=0.9322  token-acc=0.9768
K=0.001   F1=0.9322  token-acc=0.9768
K=0.0001  F1=0.9322  token-acc=0.9768
